# 00 · Setup and first trace

Phase 0 smoke test: make **one** Claude call and confirm it lands in LangSmith with the full
plan §9.4 metadata, before any agent or eval runs.

- Goes to the LangSmith project `nestling-setup`, so the `nestling-ask-baseline` and
  `nestling-ask-improved` projects stay clean for the Week 4 comparison.
- The prompt is deliberately non-medical: there is no Guardian or source library yet, so this
  call must not produce an unsourced health answer (constitution II).
- No personal data goes into the prompt or the metadata (constitution V).
- The logic lives in `nestling/tracing.py`; this notebook only calls it.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

# Show which keys are present, never their values.
for key in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_ENDPOINT"):
    print(f"{key:20} {'set' if os.getenv(key) else 'MISSING'}")
print(f"{'LANGSMITH_TRACING':20} {os.getenv('LANGSMITH_TRACING')}")

python-dotenv could not parse statement starting at line 6


ANTHROPIC_API_KEY    set
LANGSMITH_API_KEY    set
LANGSMITH_ENDPOINT   set
LANGSMITH_TRACING    true


## The traced call

`wrap_anthropic` turns the Claude call into a child LLM run with token usage. The parent run
(`first_trace`) carries the §9.4 metadata. Fields that need the Guardian, Concierge or golden
set (`expected_*`, `predicted_*`, `refused`, `correct`) stay `None` until those exist; the
measured fields (`model`, `latency_ms`, `tokens_*`, `cost_inr`, `error`) are filled in by
`traced_claude_call`.

The request opts into server-side refusal fallbacks (`fallbacks="default"`): if a safety
classifier declines, the API re-runs the request on a fallback model instead of returning a
refusal. `model` in the metadata records whichever model actually answered.

In [2]:
import anthropic
from langsmith.wrappers import wrap_anthropic

from nestling.tracing import traced_claude_call

MODEL = "claude-sonnet-5-5"
PROJECT = "nestling-setup"

client = wrap_anthropic(anthropic.Anthropic())

response, run_id = traced_claude_call(
    client,
    run_name="first_trace",
    project_name=PROJECT,
    metadata={
        "case_id": "SETUP-00",
        "dataset_version": None,  # golden_v1 is built in Phase 1
        "scenario_type": "setup",
        "pipeline_version": "setup",
        "prompt_version": "setup-v0",
        "sources_cited": [],
    },
    model=MODEL,
    max_tokens=1024,
    output_config={"effort": "low"},
    betas=["server-side-fallback-2026-07-01"],
    fallbacks="default",
    system=(
        "You are Nestling, a calm, warm companion for first-time parents. "
        "This is a setup check: reply in one short, kind sentence and give no medical advice."
    ),
    messages=[{"role": "user", "content": "Say hello to a parent who has just opened Nestling."}],
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "text":
        print(block.text)

stop_reason: end_turn
Hello and welcome to Nestling, I'm so glad you're here, and I'm happy to keep you company on this new adventure. 💛


## Verify the trace

Read the run back from LangSmith and check that every §9.4 key is stored.

In [3]:
from langsmith import Client

from nestling.tracing import METADATA_KEYS, fetch_run

ls_client = Client()
run = fetch_run(ls_client, run_id)
stored = run.extra.get("metadata", {})

missing = [key for key in METADATA_KEYS if key not in stored]
print("Trace URL:", ls_client.get_run_url(run=run, project_name=PROJECT))
print("Missing §9.4 keys:", missing or "none")
print()
for key in METADATA_KEYS:
    print(f"  {key:20} {stored.get(key)!r}")

E:\maai-projects\nestling\nestling\tracing.py:111: LangSmithDeprecationWarning: read_run() is deprecated and will be removed after Jan 31, 2027. Use client.runs.retrieve() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-retrieve for the migration guide.
  run = ls_client.read_run(run_id)


C:\Users\valli\AppData\Local\Temp\ipykernel_12448\2242195880.py:10: LangSmithDeprecationWarning: get_run_url() is deprecated and will be removed after Jan 31, 2027. Use client.runs.get_url() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-get-url for the migration guide.
  print("Trace URL:", ls_client.get_run_url(run=run, project_name=PROJECT))


Trace URL: https://smith.langchain.com/o/20d8f2fd-49d5-4daa-be0a-340ae1552592/projects/p/a4be333e-b32a-406e-b1a1-9efd42991ff2/r/3d1a7c50-f1bf-4da9-9ce2-b947032ae467?poll=true
Missing §9.4 keys: none

  case_id              'SETUP-00'
  dataset_version      None
  scenario_type        'setup'
  pipeline_version     'setup'
  prompt_version       'setup-v0'
  model                'claude-sonnet-5-5'
  expected_escalate    None
  predicted_escalate   None
  expected_route       None
  predicted_route      None
  refused              None
  sources_cited        []
  correct              None
  latency_ms           2691
  tokens_in            75
  tokens_out           46
  cost_inr             0.0537
  error                None
